# IMG_DS Linear Transformer Dense Baseline with ONNX Export

## Stage 1: Software Dense Baseline

**Current scope:** Dense software baseline + ONNX export only.

**Not included:** quantization, QONNX, sparse attention, HLS, Vivado, bitstream, PYNQ-Z2 deployment.

---

## Paper Background

1. This experiment uses the **IMG_DS dataset** from the **ViT4Mal** paper.
2. IMG_DS is an **IoT-Android mobile malware image dataset**. Original images are RGB, ~299x299x3.
3. ViT4Mal uses **dense ViT attention** on **PYNQ-Z1** with 16-bit quantization, loop pipelining, array partitioning.
4. ViT4Mal Table 13 (selected):
   - **2 encoders & 4 heads**: Acc 93.89%, BRAM 84, DSP 69, LUT 16079, FF 16356, 1.87W, 1.48s
   - **1 encoder & 2 heads**: Acc 92.41%, BRAM 92, DSP 69, LUT 16545, FF 16911, 1.89W, 0.81s
5. **This stage** only establishes the **Linear Transformer dense software + ONNX baseline**.
6. Subsequent stages: 4/8/16/32bit quantization, QONNX, sparse attention, HLS, PYNQ-Z2 deployment.

---

## Stage 1 Configuration

| Parameter | Value |
|-----------|-------|
| resize_size | 32 |
| patch_size | 8 |
| color_mode | grayscale |
| seq_len | 16 |
| input_dim | 64 |
| d_model | 16 |
| dim_feedforward | 32 |
| num_layers | 1 |
| dropout | 0.1 |
| num_classes | 2 |
| loss | CrossEntropyLoss (class_weight) |
| optimizer | AdamW (lr=0.001, wd=0.0001) |
| batch_size | 128 |
| epochs | 30 |
| seed | 42 |

**Pipeline:** IMG_DS image -> grayscale -> resize 32x32 -> 8x8 patches -> 16 tokens x 64 dims -> Linear Transformer -> [batch, 2] logits

## 1. Path & Environment Check

In [ ]:
from pathlib import Path
import sys, os, subprocess

PROJECT_ROOT = Path("/home/cym/prj2/finn/notebooks/icl_thesis-master")
DATA_DIR = PROJECT_ROOT / "data/IMG_DS"
SRC_FILE = Path("/home/cym/prj2/finn/src/transformer.py")
EXP_DIR = PROJECT_ROOT / "experiments/imgds_linear_sparse"
NOTEBOOK_DIR = Path("/home/cym/prj2/finn/notebooks/end2end_example/cybersecurity")

for d in ["outputs", "checkpoints", "models", "onnx", "reports", "figures", "logs", "package"]:
    (EXP_DIR / d).mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Version:", sys.version.split()[0])
print("DATA_DIR exists:", DATA_DIR.exists())
print("Benign dir:", (DATA_DIR / "Benign").exists())
print("Malware dir:", (DATA_DIR / "Malware").exists())

# Check packages, install missing via kernel pip
for pkg_name, import_name in [("torch", "torch"), ("onnx", "onnx"), ("onnxruntime", "onnxruntime"), ("onnxscript", "onnxscript"), ("tabulate", "tabulate")]:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, "__version__", "?")
        print(f"  [OK] {pkg_name} == {ver}")
    except ImportError:
        print(f"  [INSTALL] {pkg_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])
        __import__(import_name)
        print(f"  [OK] {pkg_name} installed")

# QONNX (check only)
try:
    import qonnx
    print(f"  [OK] qonnx available (not used in Stage 1)")
except ImportError:
    print(f"  [INFO] qonnx not available (not needed for Stage 1)")

import torch
print(f"CUDA: {torch.cuda.is_available()}")

## 2. Imports & Random Seed

In [ ]:
import os, sys, time, json, zipfile, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, str(PROJECT_ROOT))

# Import transformer components from src/transformer.py
import importlib.util
tf_path = Path("/home/cym/prj2/finn/src/transformer.py")
spec = importlib.util.spec_from_file_location("transformer", tf_path)
tf_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tf_module)
PositionalEncoding = tf_module.PositionalEncoding
LinearTransformerEncoder = tf_module.LinearTransformerEncoder
print("Loaded transformer building blocks from", tf_path)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seed: {SEED}, Device: {DEVICE}")

## 3. IMG_DS Dataset Audit

In [ ]:
benign_dir = DATA_DIR / "Benign"
malware_dir = DATA_DIR / "Malware"
benign_files = sorted([f for f in benign_dir.iterdir() if f.is_file()])
malware_files = sorted([f for f in malware_dir.iterdir() if f.is_file()])
n_benign, n_malware = len(benign_files), len(malware_files)
print(f"Benign: {n_benign}, Malware: {n_malware}, Total: {n_benign+n_malware}")
print(f"Imbalance: {n_malware/n_benign:.2f}x")

# Extensions
be = Counter(f.suffix.lower() for f in benign_files)
me = Counter(f.suffix.lower() for f in malware_files)
print(f"Benign exts: {dict(be)}")
print(f"Malware exts: {dict(me)}")

# Sample audit
MAX_SAMPLE = 200
audit_rows = []
for label, flist in [("Benign", benign_files), ("Malware", malware_files)]:
    sample = np.random.choice(flist, min(MAX_SAMPLE, len(flist)), replace=False)
    for fp in sample:
        try:
            img = Image.open(fp)
            w, h = img.size; mode = img.mode; ch = len(img.getbands())
            corrupt = False
            audit_rows.append({"label":label, "file":fp.name, "w":w, "h":h, "mode":mode, "ch":ch, "corrupt":False})
        except:
            audit_rows.append({"label":label, "file":fp.name, "w":-1, "h":-1, "mode":"ERR", "ch":-1, "corrupt":True})
df_audit = pd.DataFrame(audit_rows)
print(f"Corrupt: {df_audit['corrupt'].sum()}")
print(df_audit.groupby("label")[["w","h","ch"]].describe())
df_audit.to_csv(EXP_DIR / "reports/imgds_audit.csv", index=False)

# Markdown report
md = f"""# IMG_DS Audit
- Benign: {n_benign}, Malware: {n_malware}, Total: {n_benign+n_malware}
- Formats: Benign={dict(be)}, Malware={dict(me)}
- Corrupt in sample: {df_audit['corrupt'].sum()}
- Suitable for grayscale: Yes
"""
(EXP_DIR / "reports/imgds_audit.md").write_text(md)
display(df_audit.head(10))

## 4. Image to Patch Sequence

In [ ]:
RESIZE_SIZE = 32; PATCH_SIZE = 8
SEQ_LEN = (RESIZE_SIZE // PATCH_SIZE) ** 2  # 16
INPUT_DIM = PATCH_SIZE * PATCH_SIZE  # 64
print(f"resize={RESIZE_SIZE}, patch={PATCH_SIZE}, seq_len={SEQ_LEN}, dim={INPUT_DIM}")

def load_image_to_patch_sequence(path, resize_size=32, patch_size=8):
    img = Image.open(path).convert("L")
    img = img.resize((resize_size, resize_size), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    n = resize_size // patch_size
    patches = []
    for i in range(n):
        for j in range(n):
            p = arr[i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size]
            patches.append(p.flatten())
    return np.stack(patches, axis=0)

all_X, all_y, all_paths = [], [], []
for lbl, flist in [(0, benign_files), (1, malware_files)]:
    bad = 0
    for i, fp in enumerate(flist):
        try:
            all_X.append(load_image_to_patch_sequence(fp, RESIZE_SIZE, PATCH_SIZE))
            all_y.append(lbl); all_paths.append(str(fp))
        except: bad += 1
        if (i+1) % 2000 == 0: print(f"  {i+1}/{len(flist)}...")
    print(f"Label {lbl}: {len(flist)-bad} ok, {bad} bad")

X = np.stack(all_X, 0).astype(np.float32)
y = np.array(all_y, dtype=np.int64)
paths_arr = np.array(all_paths)
print(f"X: {X.shape}, y: {y.shape}, B:{(y==0).sum()}, M:{(y==1).sum()}")

# Stratified split 70/10/20
idx = np.arange(len(y))
train_idx, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
val_idx, test_idx = train_test_split(tmp, test_size=2/3, stratify=y[tmp], random_state=SEED)
for name, ix in [("train",train_idx),("val",val_idx),("test",test_idx)]:
    print(f"{name}: {len(ix)} (B={(y[ix]==0).sum()}, M={(y[ix]==1).sum()})")
    np.savez_compressed(EXP_DIR/f"outputs/imgds_r32_p8_{name}.npz", X=X[ix], y=y[ix], paths=paths_arr[ix])

# 200 eval samples
eval_ix = test_idx[:200]
np.savez_compressed(EXP_DIR/"outputs/imgds_r32_p8_fpga_eval_200.npz", X=X[eval_ix], y=y[eval_ix], paths=paths_arr[eval_ix])

split_md = f"""# Patch Split Report
- resize={RESIZE_SIZE}, patch={PATCH_SIZE}, seq_len={SEQ_LEN}, dim={INPUT_DIM}
- Train: {X[train_idx].shape}, Val: {X[val_idx].shape}, Test: {X[test_idx].shape}
- Stage 1 uses 32x32 grayscale for fast baseline. Later: 64x64/128x128.
"""
(EXP_DIR/"reports/imgds_patch_split_report.md").write_text(split_md)
print("Splits saved.")

## 5. DataLoaders

In [ ]:
BATCH_SIZE = 128
tr = np.load(EXP_DIR/"outputs/imgds_r32_p8_train.npz")
va = np.load(EXP_DIR/"outputs/imgds_r32_p8_val.npz")
te = np.load(EXP_DIR/"outputs/imgds_r32_p8_test.npz")
X_tr = torch.from_numpy(tr["X"]).float(); y_tr = torch.from_numpy(tr["y"]).long()
X_va = torch.from_numpy(va["X"]).float(); y_va = torch.from_numpy(va["y"]).long()
X_te = torch.from_numpy(te["X"]).float(); y_te = torch.from_numpy(te["y"]).long()
train_dl = DataLoader(TensorDataset(X_tr,y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(TensorDataset(X_va,y_va), batch_size=BATCH_SIZE, shuffle=False)
test_dl = DataLoader(TensorDataset(X_te,y_te), batch_size=BATCH_SIZE, shuffle=False)
print(f"Train: {len(train_dl)} batches, Val: {len(val_dl)}, Test: {len(test_dl)}")
for xb, yb in train_dl:
    print(f"Batch: x={xb.shape}, y={yb.shape}, range=[{xb.min():.3f},{xb.max():.3f}]")
    break

## 6. Model: LinearIMGDSDenseBaseline

In [ ]:
class LinearIMGDSDenseBaseline(nn.Module):
    def __init__(self, input_dim=64, seq_len=16, d_model=16, dim_feedforward=32,
                 num_layers=1, dropout=0.1, num_classes=2):
        super().__init__()
        self.input_dim = input_dim; self.seq_len = seq_len
        self.front_linear = nn.Linear(input_dim, d_model)
        self.pe = PositionalEncoding(d_model, max_len=seq_len)
        self.encoder = LinearTransformerEncoder(num_layers=num_layers,
            input_dim=d_model, dim_feedforward=dim_feedforward, dropout=dropout,
            enable_layer_norm=True, feature_map="elu")
        self.final = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.front_linear(x)
        x = self.pe(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.final(x)

model = LinearIMGDSDenseBaseline(INPUT_DIM, SEQ_LEN, 16, 32, 1, 0.1, 2).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}, Size: {n_params*4/1024:.1f} KB")
dummy = torch.randn(1,16,64).to(DEVICE)
with torch.no_grad():
    print(f"Input: {dummy.shape} -> Output: {model(dummy).shape}")

## 7. Training & Evaluation Functions

In [ ]:
def compute_metrics(y_true, y_pred, y_score=None):
    y_true, y_pred = np.asarray(y_true).flatten(), np.asarray(y_pred).flatten()
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    m = {"accuracy": accuracy_score(y_true,y_pred), "precision": precision_score(y_true,y_pred,zero_division=0),
         "recall": recall_score(y_true,y_pred,zero_division=0), "f1": f1_score(y_true,y_pred,zero_division=0),
         "tn":int(tn), "fp":int(fp), "fn":int(fn), "tp":int(tp)}
    if y_score is not None:
        try: m["auc"] = roc_auc_score(y_true, np.asarray(y_score).flatten())
        except: m["auc"] = float("nan")
    else: m["auc"] = float("nan")
    return m

@torch.no_grad()
def evaluate(model, dl, device=DEVICE):
    model.eval(); yt, yp, ys = [], [], []
    for xb, yb in dl:
        logits = model(xb.to(device))
        probs = F.softmax(logits, -1)
        yt.append(yb.numpy()); yp.append(logits.argmax(-1).cpu().numpy()); ys.append(probs[:,1].cpu().numpy())
    return compute_metrics(np.concatenate(yt), np.concatenate(yp), np.concatenate(ys))

print("Ready.")

## 8. Train Dense Baseline

In [ ]:
torch.manual_seed(SEED); np.random.seed(SEED)
model = LinearIMGDSDenseBaseline(INPUT_DIM, SEQ_LEN, 16, 32, 1, 0.1, 2).to(DEVICE)
cw = compute_class_weight("balanced", classes=np.array([0,1]), y=y_tr.numpy())
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

CKPT_PATH = EXP_DIR / "checkpoints/best_linear_imgds_dense_r32_p8.pt"
EPOCHS = 30; history = []; best_f1 = -1.0; best_ep = 0
print(f"Training {EPOCHS} epochs, class_weights={cw}")
t0 = time.time()
for ep in range(1, EPOCHS+1):
    model.train(); tl = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); opt.step()
        tl += loss.item() * xb.size(0)
    tl /= len(train_dl.dataset)
    vm = evaluate(model, val_dl)
    history.append({"epoch":ep, "train_loss":tl, **{f"val_{k}":v for k,v in vm.items()}})
    if vm["f1"] > best_f1:
        best_f1 = vm["f1"]; best_ep = ep
        torch.save({"epoch":ep, "model_state_dict":model.state_dict(), "val_f1":best_f1}, CKPT_PATH)
    print(f"Epoch {ep:3d}: loss={tl:.4f}, val_acc={vm['accuracy']:.4f}, val_f1={vm['f1']:.4f}, val_auc={vm['auc']:.4f}")

print(f"\nBest: epoch={best_ep}, f1={best_f1:.4f}, time={time.time()-t0:.0f}s")
pd.DataFrame(history).to_csv(EXP_DIR/"reports/train_metrics.csv", index=False)

# Test
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
test_m = evaluate(model, test_dl)
print(f"\nTest: acc={test_m['accuracy']:.4f}, prec={test_m['precision']:.4f}, rec={test_m['recall']:.4f}, f1={test_m['f1']:.4f}, auc={test_m['auc']:.4f}")
print(f"CM: TN={test_m['tn']}, FP={test_m['fp']}, FN={test_m['fn']}, TP={test_m['tp']}")
pd.DataFrame([test_m]).to_csv(EXP_DIR/"reports/test_metrics.csv", index=False)
pd.DataFrame({"":["Actual_Benign","Actual_Malware"], "Pred_Benign":[test_m["tn"],test_m["fn"]], "Pred_Malware":[test_m["fp"],test_m["tp"]]}).to_csv(EXP_DIR/"reports/confusion_matrix.csv", index=False)
param_count = sum(p.numel() for p in model.parameters())
mb = param_count * 4 / (1024*1024)

# Plot
h = pd.DataFrame(history)
fig, ax = plt.subplots(1,3,figsize=(15,4))
ax[0].plot(h["epoch"], h["train_loss"], label="Train"); ax[0].plot(h["epoch"], h["val_loss"], label="Val"); ax[0].legend(); ax[0].set_title("Loss")
ax[1].plot(h["epoch"], h["val_accuracy"], label="Acc"); ax[1].plot(h["epoch"], h["val_f1"], label="F1"); ax[1].legend(); ax[1].set_title("Metrics")
ax[2].plot(h["epoch"], h["val_auc"], label="AUC"); ax[2].legend(); ax[2].set_title("AUC")
plt.tight_layout(); plt.savefig(EXP_DIR/"figures/training_curves.png", dpi=120); plt.show()

report_md = f"""# Dense Baseline Report
- Stage: software dense baseline, no HLS/FPGA/PYNQ
- Params: {param_count:,}, Size: {mb:.2f} MB
- Test: acc={test_m['accuracy']:.4f}, f1={test_m['f1']:.4f}, auc={test_m['auc']:.4f}
- CM: TN={test_m['tn']}, FP={test_m['fp']}, FN={test_m['fn']}, TP={test_m['tp']}
- Class weight: Benign={cw[0]:.2f}, Malware={cw[1]:.2f}
"""
(EXP_DIR/"reports/linear_imgds_dense_baseline_report.md").write_text(report_md)
print("Report saved.")

## 9. ONNX Export

In [ ]:
import onnx, onnxruntime as ort
ONNX_PATH = EXP_DIR / "onnx/linear_imgds_dense_r32_p8.onnx"

model.eval(); model.to("cpu")
dummy = torch.randn(1, 16, 64, dtype=torch.float32)
torch.onnx.export(model, dummy, str(ONNX_PATH), opset_version=17,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input":{0:"batch"}, "logits":{0:"batch"}})

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
print(f"ONNX: {ONNX_PATH} ({os.path.getsize(ONNX_PATH)/1024:.1f} KB), checker PASSED")

export_md = f"""# ONNX Export Report
- Checkpoint: {CKPT_PATH}
- ONNX: {ONNX_PATH} ({os.path.getsize(ONNX_PATH)/1024:.1f} KB)
- Input: [batch, 16, 64], Output: [batch, 2], Opset: 17, Dynamic batch: Yes
"""
(EXP_DIR/"reports/onnx_export_report.md").write_text(export_md)
print("Export report saved.")

## 10. ONNX Runtime Validation

In [ ]:
sess = ort.InferenceSession(str(ONNX_PATH))
N_CMP = min(200, len(X_te))
X_cmp = X_te[:N_CMP].numpy(); y_cmp = y_te[:N_CMP].numpy()

# PyTorch
with torch.no_grad():
    pt_logits = model(torch.from_numpy(X_cmp)).numpy()
pt_probs = F.softmax(torch.tensor(pt_logits), -1).numpy()
pt_preds = pt_logits.argmax(-1)

# ONNX
ort_out = sess.run(None, {"input": X_cmp})
ort_logits = ort_out[0]
ort_probs = F.softmax(torch.tensor(ort_logits), -1).numpy()
ort_preds = ort_logits.argmax(-1)

abs_err = np.abs(pt_logits - ort_logits)
diff = {"max_abs_error": float(abs_err.max()), "mean_abs_error": float(abs_err.mean()),
        "rmse": float(np.sqrt(np.mean((pt_logits-ort_logits)**2))),
        "allclose_1e-5": bool(np.allclose(pt_logits, ort_logits, atol=1e-5, rtol=1e-4))}
print(f"Diff: max={diff['max_abs_error']:.8f}, mean={diff['mean_abs_error']:.8f}, rmse={diff['rmse']:.8f}, allclose={diff['allclose_1e-5']}")

ort_m = compute_metrics(y_cmp, ort_preds, ort_probs[:,1])
print(f"ONNX metrics: acc={ort_m['accuracy']:.4f}, f1={ort_m['f1']:.4f}, auc={ort_m['auc']:.4f}")

# Graph stats
ops = Counter(n.op_type for n in onnx_model.graph.node)
print(f"ONNX graph: {len(onnx_model.graph.node)} nodes, {len(onnx_model.graph.initializer)} initializers")
print(f"Ops: {dict(ops)}")

pd.DataFrame([ort_m]).to_csv(EXP_DIR/"reports/onnx_metrics.csv", index=False)
pd.DataFrame([diff]).to_csv(EXP_DIR/"reports/onnx_pytorch_diff.csv", index=False)
pd.DataFrame([{"node_count":len(onnx_model.graph.node), "initializer_count":len(onnx_model.graph.initializer), **dict(ops)}]).to_csv(EXP_DIR/"reports/onnx_graph_stats.csv", index=False)

val_md = f"""# ONNX Validation Report
- Checker: PASSED, Samples: {N_CMP}
- max_abs_error: {diff['max_abs_error']:.8f}, allclose: {diff['allclose_1e-5']}
- ONNX metrics: acc={ort_m['accuracy']:.4f}, f1={ort_m['f1']:.4f}
"""
(EXP_DIR/"reports/onnx_validation_report.md").write_text(val_md)
print("Validation report saved.")

## 11. QONNX Status Check (no quantization export)

In [ ]:
try:
    import qonnx
    q_avail = True
    print("QONNX available (not used in Stage 1)")
except ImportError:
    q_avail = False
    print("QONNX not available (not needed for Stage 1)")
json.dump({"qonnx_available": q_avail, "stage": "Stage 1 - no quantization"},
          open(EXP_DIR/"reports/qonnx_readiness.json","w"), indent=2)
print("QONNX readiness saved.")

## 12. PyTorch CPU Inference Time

In [ ]:
N_TIME = min(1000, len(X_te))
X_tm = X_te[:N_TIME].to("cpu")
model.eval(); model.to("cpu")
for _ in range(10): _ = model(X_tm[:1])  # warmup
times = []
with torch.no_grad():
    for i in range(N_TIME):
        t0 = time.perf_counter(); _ = model(X_tm[i:i+1]); t1 = time.perf_counter()
        times.append((t1-t0)*1e6)
times = np.array(times)
cpu_s = {"n_samples": N_TIME, "mean_us": float(times.mean()), "std_us": float(times.std()),
         "p50_us": float(np.percentile(times,50)), "p90_us": float(np.percentile(times,90)),
         "p99_us": float(np.percentile(times,99)), "note": "PyTorch CPU only, NOT PYNQ-Z2"}
print(f"CPU: mean={cpu_s['mean_us']:.1f}us, p50={cpu_s['p50_us']:.1f}us, p99={cpu_s['p99_us']:.1f}us")
pd.DataFrame([cpu_s]).to_csv(EXP_DIR/"reports/cpu_inference_time.csv", index=False)

## 13. ViT4Mal Reference Baselines

In [ ]:
vit4mal_data = [
    ["ViT4Mal","IMG_DS","PYNQ-Z1","4 enc & 4 heads",92.16,84,69,16135,16364,1.89,2.82,"dense ViT"],
    ["ViT4Mal","IMG_DS","PYNQ-Z1","4 enc & 2 heads",93.16,92,69,16200,16390,1.90,2.81,"dense ViT"],
    ["ViT4Mal","IMG_DS","PYNQ-Z1","2 enc & 4 heads",93.89,84,69,16079,16356,1.87,1.48,"dense ViT recommended"],
    ["ViT4Mal","IMG_DS","PYNQ-Z1","2 enc & 2 heads",93.86,92,69,16219,16393,1.88,1.45,"dense ViT"],
    ["ViT4Mal","IMG_DS","PYNQ-Z1","1 enc & 4 heads",92.47,84,69,16029,16358,1.85,0.87,"dense ViT"],
    ["ViT4Mal","IMG_DS","PYNQ-Z1","1 enc & 2 heads",92.41,92,69,16545,16911,1.89,0.81,"dense ViT fastest"],
]
cols = ["method","dataset","board","architecture","accuracy","BRAM","DSP","LUT","FF","power_w","inference_time_s","notes"]
df_vit4mal = pd.DataFrame(vit4mal_data, columns=cols)
display(df_vit4mal)
df_vit4mal.to_csv(EXP_DIR/"reports/reference_baselines.csv", index=False)

# Manual markdown (no tabulate dependency)
md_lines = ["# ViT4Mal Reference Baselines (Table 13)\n"]
md_lines.append("|" + "|".join(cols) + "|")
md_lines.append("|" + "|".join(["---"]*len(cols)) + "|")
for row in vit4mal_data:
    md_lines.append("|" + "|".join(str(x) for x in row) + "|")
md_lines.append("\nNote: ViT4Mal dense ViT on PYNQ-Z1. Our Linear Transformer targets lower FPGA resource usage.\n")
(EXP_DIR/"reports/reference_baselines.md").write_text("\n".join(md_lines))
print("Reference baselines saved.")

## 14. Stage 1 Summary Table

In [ ]:
df_t = pd.read_csv(EXP_DIR/"reports/test_metrics.csv")
df_om = pd.read_csv(EXP_DIR/"reports/onnx_metrics.csv")
df_d = pd.read_csv(EXP_DIR/"reports/onnx_pytorch_diff.csv")
df_ct = pd.read_csv(EXP_DIR/"reports/cpu_inference_time.csv")

our = {"method":"Ours-LinearTransformer-Dense", "dataset":"IMG_DS",
    "stage":"software_dense_onnx_baseline", "input_mode":"grayscale",
    "resize_size":RESIZE_SIZE, "patch_size":PATCH_SIZE, "seq_len":SEQ_LEN, "input_dim":INPUT_DIM,
    "d_model":16, "num_layers":1, "dim_feedforward":32, "quant_bits":32, "sparsity":0,
    "accuracy":df_t["accuracy"][0], "precision":df_t["precision"][0], "recall":df_t["recall"][0],
    "f1":df_t["f1"][0], "auc":df_t["auc"][0],
    "onnx_accuracy":df_om["accuracy"][0], "onnx_precision":df_om["precision"][0],
    "onnx_recall":df_om["recall"][0], "onnx_f1":df_om["f1"][0], "onnx_auc":df_om["auc"][0],
    "onnx_max_abs_error":df_d["max_abs_error"][0], "onnx_mean_abs_error":df_d["mean_abs_error"][0],
    "onnx_rmse":df_d["rmse"][0], "onnx_allclose":df_d["allclose_1e-5"][0],
    "tn":int(df_t["tn"][0]), "fp":int(df_t["fp"][0]), "fn":int(df_t["fn"][0]), "tp":int(df_t["tp"][0]),
    "parameter_count":param_count, "model_size_mb":round(mb,3),
    "onnx_model_path":str(ONNX_PATH),
    "BRAM":"NA", "DSP":"NA", "LUT":"NA", "FF":"NA",
    "hls_latency_cycles":"NA", "pynq_kernel_us":"NA",
    "cpu_model_inference_us_mean":df_ct["mean_us"][0],
    "notes":"Dense software baseline with ONNX; no HLS/PYNQ yet."}

vit_rows = []
for _, r in df_vit4mal.iterrows():
    vit_rows.append({"method":r["method"], "dataset":r["dataset"], "stage":"published",
        "input_mode":"RGB", "resize_size":299, "patch_size":"?", "seq_len":"?", "input_dim":"?",
        "d_model":"?", "num_layers":"?", "dim_feedforward":"?", "quant_bits":16, "sparsity":0,
        "accuracy":r["accuracy"], "precision":"NA", "recall":"NA", "f1":"NA", "auc":"NA",
        "onnx_accuracy":"NA", "onnx_precision":"NA", "onnx_recall":"NA", "onnx_f1":"NA", "onnx_auc":"NA",
        "onnx_max_abs_error":"NA", "onnx_mean_abs_error":"NA", "onnx_rmse":"NA", "onnx_allclose":"NA",
        "tn":"NA", "fp":"NA", "fn":"NA", "tp":"NA",
        "parameter_count":"NA", "model_size_mb":"NA", "onnx_model_path":"NA",
        "BRAM":r["BRAM"], "DSP":r["DSP"], "LUT":r["LUT"], "FF":r["FF"],
        "hls_latency_cycles":"?", "pynq_kernel_us":r["inference_time_s"]*1e6,
        "cpu_model_inference_us_mean":"NA",
        "notes":f"ViT4Mal {r['architecture']} - {r['notes']}"})

all_rows = [our] + vit_rows
df_summary = pd.DataFrame(all_rows)
display(df_summary)
df_summary.to_csv(EXP_DIR/"reports/stage1_summary_table.csv", index=False)

# Manual markdown
cols_s = list(our.keys())
md_s = ["# Stage 1 Summary Table\n", "|" + "|".join(cols_s) + "|", "|" + "|".join(["---"]*len(cols_s)) + "|"]
for row in all_rows:
    md_s.append("|" + "|".join(str(row.get(c,"")) for c in cols_s) + "|")
md_s.append("\n## Notes\n- Ours: float32 software/ONNX baseline, no FPGA/PYNQ.\n- ViT4Mal: 16-bit quantized, PYNQ-Z1.\n")
(EXP_DIR/"reports/stage1_summary_table.md").write_text("\n".join(md_s))
print("Summary table saved.")

## 15. Generate Stage 1 Package

In [ ]:
ZIP_PATH = EXP_DIR / "package/imgds_linear_dense_onnx_stage1_package.zip"
files = []
# Notebook
nb_path = Path("/home/cym/prj2/finn/notebooks/icl_thesis-master/notebooks/IMGDS_Linear_Dense_ONNX.ipynb")
if nb_path.exists(): files.append(("IMGDS_Linear_Dense_ONNX.ipynb", nb_path))
# Checkpoint
if CKPT_PATH.exists(): files.append(("best_linear_imgds_dense_r32_p8.pt", CKPT_PATH))
# ONNX
if ONNX_PATH.exists(): files.append(("linear_imgds_dense_r32_p8.onnx", ONNX_PATH))
# FPGA eval
fp = EXP_DIR/"outputs/imgds_r32_p8_fpga_eval_200.npz"
if fp.exists(): files.append(("imgds_r32_p8_fpga_eval_200.npz", fp))
# Reports
for rf in (EXP_DIR/"reports").glob("*"):
    if rf.is_file(): files.append((f"reports/{rf.name}", rf))
# Training curves
fc = EXP_DIR/"figures/training_curves.png"
if fc.exists(): files.append(("training_curves.png", fc))

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for an, fp in files: zf.write(fp, an)
print(f"Package: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1024/1024:.1f} MB, {len(files)} files)")

mani = ["# Stage 1 Package - Software/ONNX Baseline Only", "", "## WARNING: No bitstream, hwh, or PYNQ-Z2 runtime.", ""]
for an, fp in files: mani.append(f"- {an}")
(EXP_DIR/"reports/package_manifest.md").write_text("\n".join(mani))
print("Package manifest saved.")

## Stage 1 Complete

**Completed:**
- IMG_DS dataset audit
- Image -> grayscale -> resize 32x32 -> 8x8 patches -> [16, 64] tokens
- Stratified train/val/test split (70/10/20)
- Linear Transformer dense baseline training (30 epochs)
- PyTorch checkpoint saved
- Test evaluation (accuracy, precision, recall, f1, auc, confusion matrix)
- ONNX export (opset 17, dynamic batch)
- ONNX Runtime validation (PyTorch vs ONNX numerical comparison)
- QONNX availability check (no quantized export)
- CPU inference timing
- ViT4Mal reference baselines
- Stage 1 summary table
- Experiment package (.zip)

**Next:**
- Stage 2: Dense HLS baseline (if 32x32 accuracy sufficient)
- Stage 2b: Input ablation (64x64 / 128x128) if needed
- Stage 3: 4/8/16/32bit quantization + QONNX
- Stage 4: Sparse attention (0/10/30/50/70/90/100%)
- Stage 5: HLS + Vivado synthesis
- Stage 6: PYNQ-Z2 deployment

**Current stage does NOT include HLS, bitstream, hwh, or PYNQ-Z2 runtime.**

In [ ]:
# Final Summary
print("="*70)
print("STAGE 1 COMPLETE")
print("="*70)
print(f"Checkpoint: {CKPT_PATH}")
print(f"ONNX: {ONNX_PATH}")
print(f"Test: acc={test_m['accuracy']:.4f}, f1={test_m['f1']:.4f}, auc={test_m['auc']:.4f}")
print(f"ONNX: acc={ort_m['accuracy']:.4f}, f1={ort_m['f1']:.4f}, allclose={diff['allclose_1e-5']}")
print(f"CPU: mean={cpu_s['mean_us']:.1f}us, p99={cpu_s['p99_us']:.1f}us")
print(f"Package: {ZIP_PATH}")
print("\nCurrent stage does NOT include HLS, bitstream, hwh, or PYNQ-Z2 runtime.")